# Untrained CNN Project | Optimizing the Performance of the Untrained CNN Architecture

### Imports & Libraries

In [ ]:
%load_ext slurm_magic

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import scipy.ndimage as ndi
from sklearn.preprocessing import StandardScaler

### Dataset Configurations

In [ ]:
class CNNDataset_Zscore(Dataset):
    """Z-score normalization (mean=0, std=1). Baseline preprocessing."""
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = nib.load(self.paths[idx]).get_fdata().astype(np.float32)
        if img.shape != (91, 109, 91):
            img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                                 91 / img.shape[2]], order=1)
        img = (img - img.mean()) / (img.std() + 1e-6)
        return torch.from_numpy(img[None]).float()


class CNNDataset_MinMax(Dataset):
    """Min-max scaling to [0, 1]"""
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = nib.load(self.paths[idx]).get_fdata().astype(np.float32)
        if img.shape != (91, 109, 91):
            img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                                 91 / img.shape[2]], order=1)
        img_min, img_max = img.min(), img.max()
        img = (img - img_min) / (img_max - img_min + 1e-6)
        return torch.from_numpy(img[None]).float()


class CNNDataset_PercentileClip(Dataset):
    """Clip to [1st, 99th] percentile then z-score"""
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = nib.load(self.paths[idx]).get_fdata().astype(np.float32)
        if img.shape != (91, 109, 91):
            img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                                 91 / img.shape[2]], order=1)
        p1, p99 = np.percentile(img, [1, 99])
        img = np.clip(img, p1, p99)
        img = (img - img.mean()) / (img.std() + 1e-6)
        return torch.from_numpy(img[None]).float()


class CNNDataset_PercentileMinMax(Dataset):
    """Clip to [1st, 99th] percentile then min-max to [0, 1]"""
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = nib.load(self.paths[idx]).get_fdata().astype(np.float32)
        if img.shape != (91, 109, 91):
            img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                                 91 / img.shape[2]], order=1)
        p1, p99 = np.percentile(img, [1, 99])
        img = np.clip(img, p1, p99)
        img_min, img_max = img.min(), img.max()
        img = (img - img_min) / (img_max - img_min + 1e-6)
        return torch.from_numpy(img[None]).float()

### Default CNN (original MICCAI baseline)

In [ ]:
DROPOUT_RATE = 0.3 #Does not do anything model is in .eval()
 
class CNN3D_Default(nn.Module):
    def __init__(self, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.bn1, self.pool1, self.drop1 = nn.BatchNorm3d(64), nn.MaxPool3d(2), nn.Dropout3d(dropout_rate)
 
        self.conv2 = nn.Conv3d(64, 64, 3, padding=1)
        self.bn2, self.pool2, self.drop2 = nn.BatchNorm3d(64), nn.MaxPool3d(2), nn.Dropout3d(dropout_rate)
 
        self.conv3 = nn.Conv3d(64, 128, 3, padding=1)
        self.bn3, self.pool3, self.drop3 = nn.BatchNorm3d(128), nn.MaxPool3d(2), nn.Dropout3d(dropout_rate)
 
        self.conv4 = nn.Conv3d(128, 256, 3, padding=1)
        self.bn4, self.pool4, self.drop4 = nn.BatchNorm3d(256), nn.MaxPool3d(2), nn.Dropout3d(dropout_rate)
 
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(256, 512)
        self.drop5 = nn.Dropout(dropout_rate)
 
    def forward(self, x):
        x = self.drop1(self.bn1(self.pool1(F.relu(self.conv1(x)))))
        x = self.drop2(self.bn2(self.pool2(F.relu(self.conv2(x)))))
        x = self.drop3(self.bn3(self.pool3(F.relu(self.conv3(x)))))
        x = self.drop4(self.bn4(self.pool4(F.relu(self.conv4(x)))))
        return self.drop5(F.relu(self.fc1(self.global_pool(x).view(x.size(0), -1))))

### CNN3D_Spatial Pool

In [ ]:
class CNN3D_SpatialPool(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(2)
 
        self.conv2 = nn.Conv3d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(2)
 
        self.conv3 = nn.Conv3d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm3d(256)
        self.pool3 = nn.MaxPool3d(2)
 
        self.conv4 = nn.Conv3d(256, 512, 3, padding=1)
        self.bn4 = nn.BatchNorm3d(512)
        self.pool4 = nn.MaxPool3d(2)
 
        self.spatial_pool = nn.AdaptiveAvgPool3d(3)
 
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        return self.spatial_pool(x).view(x.size(0), -1)

### CNN3D_MultiScale

In [ ]:
class CNN3D_MultiScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(2)
 
        self.conv2 = nn.Conv3d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(2)
 
        self.conv3 = nn.Conv3d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm3d(256)
        self.pool3 = nn.MaxPool3d(2)
 
        self.conv4 = nn.Conv3d(256, 512, 3, padding=1)
        self.bn4 = nn.BatchNorm3d(512)
        self.pool4 = nn.MaxPool3d(2)
 
        self.pool = nn.AdaptiveAvgPool3d(2)
 
    def forward(self, x):
        x1 = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x2 = self.pool2(F.relu(self.bn2(self.conv2(x1))))
        x3 = self.pool3(F.relu(self.bn3(self.conv3(x2))))
        x4 = self.pool4(F.relu(self.bn4(self.conv4(x3))))
        return torch.cat([
            self.pool(x1).view(x.size(0), -1),
            self.pool(x2).view(x.size(0), -1),
            self.pool(x3).view(x.size(0), -1),
            self.pool(x4).view(x.size(0), -1),
        ], dim=1)

### CNN3D_Optimized

In [ ]:
class CNN3D_Optimized(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 64)
        self.down1 = nn.AvgPool3d(2)
 
        self.conv2 = nn.Conv3d(64, 128, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, 128)
        self.down2 = nn.AvgPool3d(2)
 
        self.conv3 = nn.Conv3d(128, 256, 3, padding=1)
        self.norm3 = nn.GroupNorm(8, 256)
        self.down3 = nn.AvgPool3d(2)
 
        self.conv4 = nn.Conv3d(256, 512, 3, padding=1)
        self.norm4 = nn.GroupNorm(8, 512)
        self.down4 = nn.AvgPool3d(2)
 
        self.pool = nn.AdaptiveAvgPool3d(2)
 
    def forward(self, x):
        x1 = self.down1(F.relu(self.norm1(self.conv1(x))))
        x2 = self.down2(F.relu(self.norm2(self.conv2(x1))))
        x3 = self.down3(F.relu(self.norm3(self.conv3(x2))))
        x4 = self.down4(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([
            self.pool(x1).view(x.size(0), -1),   # 64 × 8 = 512
            self.pool(x2).view(x.size(0), -1),   # 128 × 8 = 1,024
            self.pool(x3).view(x.size(0), -1),   # 256 × 8 = 2,048
            self.pool(x4).view(x.size(0), -1),   # 512 × 8 = 4,096
        ], dim=1)  # total: 7,680

### Skip-pooling: less downsampling

In [ ]:
class CNN3D_SkipPool23(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 64)
        self.down1 = nn.AvgPool3d(2)
 
        self.conv2 = nn.Conv3d(64, 128, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, 128)
 
        self.conv3 = nn.Conv3d(128, 256, 3, padding=1)
        self.norm3 = nn.GroupNorm(8, 256)
 
        self.conv4 = nn.Conv3d(256, 512, 3, padding=1)
        self.norm4 = nn.GroupNorm(8, 512)
        self.down4 = nn.AvgPool3d(2)
 
        self.pool = nn.AdaptiveAvgPool3d(2)
 
    def forward(self, x):
        x1 = self.down1(F.relu(self.norm1(self.conv1(x))))
        x2 = F.relu(self.norm2(self.conv2(x1)))
        x3 = F.relu(self.norm3(self.conv3(x2)))
        x4 = self.down4(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([
            self.pool(x1).view(x.size(0), -1),
            self.pool(x2).view(x.size(0), -1),
            self.pool(x3).view(x.size(0), -1),
            self.pool(x4).view(x.size(0), -1),
        ], dim=1)
 

### Depthwise Separable CNN

In [ ]:
class DepthwiseSepConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch)
        self.pointwise = nn.Conv3d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class CNN3D_DepthwiseSep(nn.Module):
    """Depthwise separable convolutions"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 64, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 64)
        self.down1 = nn.AvgPool3d(2)

        self.conv2 = DepthwiseSepConv3d(64, 128)
        self.norm2 = nn.GroupNorm(8, 128)
        self.down2 = nn.AvgPool3d(2)

        self.conv3 = DepthwiseSepConv3d(128, 256)
        self.norm3 = nn.GroupNorm(8, 256)
        self.down3 = nn.AvgPool3d(2)

        self.conv4 = DepthwiseSepConv3d(256, 512)
        self.norm4 = nn.GroupNorm(8, 512)
        self.down4 = nn.AvgPool3d(2)

        self.pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = self.down1(F.relu(self.norm1(self.conv1(x))))
        x2 = self.down2(F.relu(self.norm2(self.conv2(x1))))
        x3 = self.down3(F.relu(self.norm3(self.conv3(x2))))
        x4 = self.down4(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([
            self.pool(x1).view(x.size(0), -1),
            self.pool(x2).view(x.size(0), -1),
            self.pool(x3).view(x.size(0), -1),
            self.pool(x4).view(x.size(0), -1),
        ], dim=1)

### 3 Channels CNN (Original + Unsharp + Sobel)

In [ ]:
#Helper Functions

def load_and_resize(path):
    img = nib.load(path).get_fdata().astype(np.float32)
    if img.shape != (91, 109, 91):
        img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                              91 / img.shape[2]], order=1)
    return img

def clip_minmax(img):
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img_min, img_max = img.min(), img.max()
    return (img - img_min) / (img_max - img_min + 1e-6)

def norm(ch):
    return (ch - ch.min()) / (ch.max() - ch.min() + 1e-6)

def sobel_mag(img):
    sx = ndi.sobel(img, axis=0)
    sy = ndi.sobel(img, axis=1)
    sz = ndi.sobel(img, axis=2)
    return np.sqrt(sx**2 + sy**2 + sz**2)

def unsharp(img, alpha=1.0, sigma=1.0):
    blurred = ndi.gaussian_filter(img, sigma=sigma)
    return np.clip(img + alpha * (img - blurred), 0, 1)

# Dataset

class DS_3ch(Dataset):
    """3ch: orig + unsharp(1,1) + Sobel."""
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = clip_minmax(load_and_resize(self.paths[idx]))
        sharp = unsharp(img, 1.0, 1.0)
        edges = norm(sobel_mag(img))
        return torch.from_numpy(np.stack([img, sharp, edges], axis=0)).float()

# Model

class DepthwiseSepConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch)
        self.pointwise = nn.Conv3d(in_ch, out_ch, 1)
    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class CNN3D_DepthwiseSep(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, 64)
        self.down1 = nn.AvgPool3d(2)
        self.conv2 = DepthwiseSepConv3d(64, 128)
        self.norm2 = nn.GroupNorm(8, 128)
        self.down2 = nn.AvgPool3d(2)
        self.conv3 = DepthwiseSepConv3d(128, 256)
        self.norm3 = nn.GroupNorm(8, 256)
        self.down3 = nn.AvgPool3d(2)
        self.conv4 = DepthwiseSepConv3d(256, 512)
        self.norm4 = nn.GroupNorm(8, 512)
        self.down4 = nn.AvgPool3d(2)
        self.pool = nn.AdaptiveAvgPool3d(2)

    def forward(self, x):
        x1 = self.down1(F.relu(self.norm1(self.conv1(x))))
        x2 = self.down2(F.relu(self.norm2(self.conv2(x1))))
        x3 = self.down3(F.relu(self.norm3(self.conv3(x2))))
        x4 = self.down4(F.relu(self.norm4(self.conv4(x3))))
        return torch.cat([
            self.pool(x1).view(x.size(0), -1),
            self.pool(x2).view(x.size(0), -1),
            self.pool(x3).view(x.size(0), -1),
            self.pool(x4).view(x.size(0), -1),
        ], dim=1)

### Weight Initialization 

In [ ]:
def init_orthogonal(model):
    for m in model.modules():
        if isinstance(m, nn.Conv3d):
            nn.init.orthogonal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
 
 
def init_xavier(model):
    for m in model.modules():
        if isinstance(m, nn.Conv3d):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

### Feature Extraction

In [ ]:
def extract_features(model, dataloader, device='cpu'):
    model.eval()
    feats = []
    with torch.no_grad():
        for x in dataloader:
            feats.append(model(x.to(device)).cpu().numpy())
    return np.vstack(feats)

### Evaluation & Random Forest Parameters

In [ ]:
def eval_classification(X, y):
    scores = []
    for seed in SEEDS:
        cv_idx, test_idx = train_test_split(
            np.arange(len(y)), test_size=int(0.1 * len(y)),
            stratify=y, random_state=seed)
        scaler = StandardScaler()
        rf = RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_split=5,
            random_state=seed, n_jobs=1, max_features='sqrt',
            class_weight='balanced')
        rf.fit(scaler.fit_transform(X[cv_idx]), y[cv_idx])
        scores.append(balanced_accuracy_score(
            y[test_idx], rf.predict(scaler.transform(X[test_idx]))))
    return np.mean(scores), np.std(scores)
 
 
def eval_regression(X, y):
    maes = []
    for seed in SEEDS:
        cv_idx, test_idx = train_test_split(
            np.arange(len(y)), test_size=int(0.1 * len(y)),
            random_state=seed)
        scaler = StandardScaler()
        rf = RandomForestRegressor(
            n_estimators=200, max_depth=6, min_samples_split=5,
            random_state=seed, n_jobs=1)
        rf.fit(scaler.fit_transform(X[cv_idx]), y[cv_idx])
        maes.append(mean_absolute_error(
            y[test_idx], rf.predict(scaler.transform(X[test_idx]))))
    return np.mean(maes), np.std(maes)
 
 
def run_all_tasks(features_all, features_male, features_female, label,
                  sex_labels, m_age, f_age):
    sex_mean, sex_std = eval_classification(features_all, sex_labels)
    male_mean, male_std = eval_regression(features_male, m_age)
    female_mean, female_std = eval_regression(features_female, f_age)
    print(f"  {label:<50} {features_all.shape[1]:>6} "
          f"{sex_mean:.3f}+/-{sex_std:.3f}   "
          f"{male_mean:.2f}+/-{male_std:.2f}   "
          f"{female_mean:.2f}+/-{female_std:.2f}")
    return {'sex': (sex_mean, sex_std),
            'male_age': (male_mean, male_std),
            'female_age': (female_mean, female_std),
            'dim': features_all.shape[1]}

### Usage of the Architecture & Best Configuration of the Architecture

In [ ]:
# Best config: CNN3D_DWSep_3channel (original + unsharp + sobel)
# torch.manual_seed(0)

# ──────────────────────────────────────────────────────────────────────
# NKI Results (958 subjects: 357 Male | 601 Female)
# ──────────────────────────────────────────────────────────────────────

# Model                          Sex Acc   Male MAE   Female MAE
# ──────────────────────────────────────────────────────────────
# CNN3D_DWSep_3ch                0.870     6.88       7.85
# CNN3D_DWSep + clip+mm          0.855     7.11       8.24
# CNN3D_Optimized + clip+mm      0.850     7.55       8.38
# CNN3D_SkipPool23 + ortho       0.854     7.71       9.32
# CNN3D_Optimized                0.857     8.45       9.65
# CNN3D_Default                  0.843    10.20      12.05

# FS Schaefer                    0.740     8.99       9.61
# FS aparc                       0.770     9.73      10.17

# Cross-dataset (CNN3D_DWSep + 3 channel vs Best FreeSurfer, all p<0.001):

# PPMI: Sex 0.811 vs 0.734, Male 5.66 vs 6.62, Female 6.86 vs 7.66
# HBN:  Sex 0.764 vs 0.734, Male 1.70 vs 2.29, Female 1.63 vs 2.08
# NKI:  Sex 0.870 vs 0.770, Male 6.88 vs 8.99, Female 7.85 vs 9.61